# Market-Driven Skill Recommendation: Local Training + Research Evidence Notebook

This notebook keeps the original modeling pipeline and adds three research-strengthening modules: (1) alpha/beta ablation for the hybrid model, (2) limited stability analysis across random seeds, and (3) qualitative + skill-level recommendation analysis.

## 1. Install packages

Chạy cell này một lần trên Colab.  
Nếu chạy local và đã cài đủ package thì có thể bỏ qua.

In [1]:
import sys

!{sys.executable} -m pip install -q setuptools
!{sys.executable} -m pip install -q "gensim>=4.3.0,<5.0.0" "joblib>=1.4.0" "networkx==3.3" "tqdm>=4.29.0" "mlxtend>=0.23.1" "matplotlib>=3.8.0"



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and global config

In [2]:
# Imports
import ast
import json
import math
import os
import random
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from mlxtend.frequent_patterns import fpgrowth, association_rules

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 200)

## 3. Data location (local or Colab)

Set `DATA_DIR` to the folder that contains the 7 generated CSV files.

In [3]:
# Set this to "." if the CSV files are in the same folder as the notebook.
DATA_DIR = "."
DATA_DIR = os.path.abspath(DATA_DIR)

INPUT_FILES = {
    "clean_df": "cleaned_model_population.csv",
    "vocab_df": "skill_vocabulary_final.csv",
    "train_df": "train_users.csv",
    "valid_df": "valid_users.csv",
    "test_df": "test_users.csv",
    "train_matrix": "fpgrowth_train_matrix.csv",
    "edges_df": "graph_edges_train.csv",
}

resolved_input_paths = {
    name: os.path.join(DATA_DIR, filename)
    for name, filename in INPUT_FILES.items()
}

missing_files = [
    path for path in resolved_input_paths.values()
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "Missing required input files:\n" + "\n".join(missing_files)
    )

resolved_input_paths

{'clean_df': 'd:\\School\\BTVN\\BDM\\fn\\improved_notebook_copies\\cleaned_model_population.csv',
 'vocab_df': 'd:\\School\\BTVN\\BDM\\fn\\improved_notebook_copies\\skill_vocabulary_final.csv',
 'train_df': 'd:\\School\\BTVN\\BDM\\fn\\improved_notebook_copies\\train_users.csv',
 'valid_df': 'd:\\School\\BTVN\\BDM\\fn\\improved_notebook_copies\\valid_users.csv',
 'test_df': 'd:\\School\\BTVN\\BDM\\fn\\improved_notebook_copies\\test_users.csv',
 'train_matrix': 'd:\\School\\BTVN\\BDM\\fn\\improved_notebook_copies\\fpgrowth_train_matrix.csv',
 'edges_df': 'd:\\School\\BTVN\\BDM\\fn\\improved_notebook_copies\\graph_edges_train.csv'}

## 4. Load data

In [4]:
clean_df = pd.read_csv(resolved_input_paths["clean_df"])
vocab_df = pd.read_csv(resolved_input_paths["vocab_df"])
train_df = pd.read_csv(resolved_input_paths["train_df"])
valid_df = pd.read_csv(resolved_input_paths["valid_df"])
test_df = pd.read_csv(resolved_input_paths["test_df"])
train_matrix = pd.read_csv(resolved_input_paths["train_matrix"])
edges_df = pd.read_csv(resolved_input_paths["edges_df"])

print("Data directory:", DATA_DIR)
print("clean_df:", clean_df.shape)
print("vocab_df:", vocab_df.shape)
print("train_df:", train_df.shape)
print("valid_df:", valid_df.shape)
print("test_df:", test_df.shape)
print("train_matrix:", train_matrix.shape)
print("edges_df:", edges_df.shape)

display(train_df.head(2))

Data directory: d:\School\BTVN\BDM\fn\improved_notebook_copies
clean_df: (24229, 13)
vocab_df: (183, 7)
train_df: (19383, 13)
valid_df: (2423, 13)
test_df: (2423, 13)
train_matrix: (19383, 183)
edges_df: (16653, 3)


,ResponseId,MainBranch,Employment,WorkExp,WorkExp_num,DevType,is_junior_0_3y,have_skills,want_skills,target_skills,n_have,n_want,n_target
0,42896,I am a developer by profession,Employed,2.0,2.0,"Developer, full-stack",True,Language:C#|Language:HTML/CSS|Language:JavaScript|Language:SQL|Language:TypeScript|Language:Visual Basic (.Net)|Database:Microsoft SQL Server|Webframe:ASP.NET|Webframe:React|DevEnv:Notepad++|DevEn...,Language:MATLAB|Language:Python|DevEnv:PyCharm,Language:MATLAB|Language:Python|DevEnv:PyCharm,12,3,3
1,36268,I am a developer by profession,Employed,8.0,8.0,"Developer, embedded applications or devices",False,Language:Assembly|Language:Bash/Shell (all shells)|Language:C|Language:Java|Language:JavaScript|Language:Perl|Language:PowerShell|Language:Python|Platform:Make|Platform:Maven (build tool)|Platform...,Language:Bash/Shell (all shells)|Language:C|Language:Python|Platform:Make|Platform:Pacman|Platform:Pip|DevEnv:Visual Studio Code,NaN,17,7,0


## 5. Validate schema and build model-ready lists

In [5]:
REQUIRED_SPLIT_COLS = ["have_skills", "want_skills", "target_skills"]
for df_name, df in [("train_df", train_df), ("valid_df", valid_df), ("test_df", test_df)]:
    missing = [c for c in REQUIRED_SPLIT_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} thiếu cột: {missing}")

if "skill" not in vocab_df.columns:
    raise ValueError("skill_vocabulary_final.csv phải có cột 'skill'")

def parse_skill_pipe_list(value):
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text == "":
        return []
    return [item.strip() for item in text.split("|") if item.strip()]

def attach_skill_lists(df):
    out = df.copy()
    out["have_list"] = out["have_skills"].apply(parse_skill_pipe_list)
    out["want_list"] = out["want_skills"].apply(parse_skill_pipe_list)
    out["target_list"] = out["target_skills"].apply(parse_skill_pipe_list)
    return out

train_df = attach_skill_lists(train_df)
valid_df = attach_skill_lists(valid_df)
test_df = attach_skill_lists(test_df)

skill_catalog = set(vocab_df["skill"].astype(str).tolist())
matrix_skill_catalog = set(train_matrix.columns.astype(str).tolist())

# Dùng intersection để đảm bảo catalog khớp hoàn toàn với feature space train
skill_catalog = skill_catalog & matrix_skill_catalog if len(skill_catalog) > 0 else matrix_skill_catalog

assert "have_list" in train_df.columns
assert "have_list" in valid_df.columns
assert "have_list" in test_df.columns

print("Skill catalog size:", len(skill_catalog))
print("Train users with non-empty target:", int((train_df["target_list"].apply(len) > 0).sum()))
print("Valid users with non-empty target:", int((valid_df["target_list"].apply(len) > 0).sum()))
print("Test users with non-empty target:", int((test_df["target_list"].apply(len) > 0).sum()))
display(train_df[["ResponseId", "have_list", "target_list"]].head(2))

Skill catalog size: 183
Train users with non-empty target: 13311
Valid users with non-empty target: 1630
Test users with non-empty target: 1702


,ResponseId,have_list,target_list
0,42896,"[Language:C#, Language:HTML/CSS, Language:JavaScript, Language:SQL, Language:TypeScript, Language:Visual Basic (.Net), Database:Microsoft SQL Server, Webframe:ASP.NET, Webframe:React, DevEnv:Notep...","[Language:MATLAB, Language:Python, DevEnv:PyCharm]"
1,36268,"[Language:Assembly, Language:Bash/Shell (all shells), Language:C, Language:Java, Language:JavaScript, Language:Perl, Language:PowerShell, Language:Python, Platform:Make, Platform:Maven (build tool...",[]


## 6. Experiment config

In [6]:
CONFIG = {
    "fp_min_support": 0.03,
    "fp_min_confidence": 0.30,
    "fp_max_antecedent_len": 3,
    # Node2Vec/Word2Vec settings are increased from the quick-run version
    # to give graph embeddings more capacity and more training signal.
    "n2v_dim": 128,
    "n2v_walk_length": 20,
    "n2v_num_walks": 40,
    "n2v_window": 5,
    "n2v_epochs": 10,
    "n2v_p": 1.0,
    "n2v_q": 0.5,
    "sim_cache_topn": 50,
    "alpha": 0.6,
    "beta": 0.4,
}

ANALYSIS_CONFIG = {
    "ablation_grid": [
        (1.0, 0.0),  # normalized rule component only inside hybrid scoring
        (0.8, 0.2),
        (0.6, 0.4),  # default hybrid
        (0.4, 0.6),
        (0.0, 1.0),  # normalized embedding component only inside hybrid scoring
    ],
    "run_stability_analysis": True,
    "stability_seeds": [42, 123, 999],
    "stability_k_values": [5, 10],
    "skill_probe_list": [
        "Language:Python",
        "Language:JavaScript",
        "Database:PostgreSQL",
        "Platform:Docker",
    ],
    "qualitative_num_cases": 15,
}

print("CONFIG =", CONFIG)
print("ANALYSIS_CONFIG =", ANALYSIS_CONFIG)


CONFIG = {'fp_min_support': 0.03, 'fp_min_confidence': 0.3, 'fp_max_antecedent_len': 3, 'n2v_dim': 128, 'n2v_walk_length': 20, 'n2v_num_walks': 40, 'n2v_window': 5, 'n2v_epochs': 10, 'n2v_p': 1.0, 'n2v_q': 0.5, 'sim_cache_topn': 50, 'alpha': 0.6, 'beta': 0.4}
ANALYSIS_CONFIG = {'ablation_grid': [(1.0, 0.0), (0.8, 0.2), (0.6, 0.4), (0.4, 0.6), (0.0, 1.0)], 'run_stability_analysis': True, 'stability_seeds': [42, 123, 999], 'stability_k_values': [5, 10], 'skill_probe_list': ['Language:Python', 'Language:JavaScript', 'Database:PostgreSQL', 'Platform:Docker'], 'qualitative_num_cases': 15}


## 7. Evaluation utilities

In [7]:
def recall_at_k(predictions, target_skills, k=10):
    target_set = set(target_skills)
    if len(target_set) == 0:
        return np.nan
    return len(set(predictions[:k]) & target_set) / len(target_set)

def hitrate_at_k(predictions, target_skills, k=10):
    target_set = set(target_skills)
    if len(target_set) == 0:
        return np.nan
    return 1.0 if len(set(predictions[:k]) & target_set) > 0 else 0.0

def skill_category(skill):
    return str(skill).split(":", 1)[0] if ":" in str(skill) else "Unknown"

def category_diversity_at_k(predictions, k=10):
    top_k_predictions = predictions[:k]
    if len(top_k_predictions) == 0:
        return 0.0
    categories = {skill_category(skill) for skill in top_k_predictions}
    return len(categories) / len(top_k_predictions)

def novelty_at_k(predictions, support_fraction_map, k=10):
    top_k_predictions = predictions[:k]
    if len(top_k_predictions) == 0 or support_fraction_map is None:
        return np.nan
    # Higher novelty means lower training popularity.
    return float(np.mean([
        -math.log2(max(support_fraction_map.get(skill, 0.0), 1e-12))
        for skill in top_k_predictions
    ]))

def average_support_at_k(predictions, support_fraction_map, k=10):
    top_k_predictions = predictions[:k]
    if len(top_k_predictions) == 0 or support_fraction_map is None:
        return np.nan
    return float(np.mean([support_fraction_map.get(skill, 0.0) for skill in top_k_predictions]))

def evaluate_predictions(df, pred_col, k=10, catalog=None, support_fraction_map=None):
    if catalog is None:
        raise ValueError("catalog must be provided to evaluate_predictions()")

    eval_df = df[df["target_list"].apply(len) > 0].copy()

    if len(eval_df) == 0:
        return {
            "UsersEvaluated": 0,
            "Recall@K": np.nan,
            "HitRate@K": np.nan,
            "CatalogCoverage": np.nan,
            "Diversity@K": np.nan,
            "Novelty@K": np.nan,
            "AvgSupport@K": np.nan,
        }

    eval_df["recall"] = eval_df.apply(
        lambda row: recall_at_k(row[pred_col], row["target_list"], k=k),
        axis=1,
    )
    eval_df["hit"] = eval_df.apply(
        lambda row: hitrate_at_k(row[pred_col], row["target_list"], k=k),
        axis=1,
    )
    eval_df["diversity"] = eval_df[pred_col].apply(lambda items: category_diversity_at_k(items, k=k))
    eval_df["novelty"] = eval_df[pred_col].apply(lambda items: novelty_at_k(items, support_fraction_map, k=k))
    eval_df["avg_support"] = eval_df[pred_col].apply(lambda items: average_support_at_k(items, support_fraction_map, k=k))

    recommended_skills = set()
    for items in eval_df[pred_col]:
        recommended_skills.update(items[:k])

    catalog_coverage = len(recommended_skills) / max(len(catalog), 1)

    return {
        "UsersEvaluated": int(len(eval_df)),
        "Recall@K": float(np.nanmean(eval_df["recall"])),
        "HitRate@K": float(np.nanmean(eval_df["hit"])),
        "CatalogCoverage": float(catalog_coverage),
        "Diversity@K": float(np.nanmean(eval_df["diversity"])),
        "Novelty@K": float(np.nanmean(eval_df["novelty"])),
        "AvgSupport@K": float(np.nanmean(eval_df["avg_support"])),
    }


## 8. Popularity baseline

In [8]:
skill_support_counter = Counter()
for user_skills in train_df["have_list"]:
    skill_support_counter.update(set(user_skills))

global_popularity_ranking = [skill for skill, _ in skill_support_counter.most_common()]
skill_support_fraction = {
    skill: count / max(len(train_df), 1)
    for skill, count in skill_support_counter.items()
}

def recommend_popularity(user_have, k=10):
    seen_skills = set(user_have)
    recommendations = []
    for skill in global_popularity_ranking:
        if skill not in seen_skills:
            recommendations.append(skill)
        if len(recommendations) == k:
            break
    return recommendations

for df in [valid_df, test_df]:
    df["pred_popularity_10"] = df["have_list"].apply(lambda skills: recommend_popularity(skills, k=10))
    df["pred_popularity_5"] = df["pred_popularity_10"].apply(lambda items: items[:5])

## 9. Train FP-Growth exactly once

In [9]:
bool_train_matrix = train_matrix.astype(bool)

fp_itemsets_df = fpgrowth(
    bool_train_matrix,
    min_support=CONFIG["fp_min_support"],
    use_colnames=True,
)

print("Frequent itemsets:", len(fp_itemsets_df))

if len(fp_itemsets_df) == 0:
    fp_rules_df = pd.DataFrame()
else:
    fp_rules_df = association_rules(
        fp_itemsets_df,
        metric="confidence",
        min_threshold=CONFIG["fp_min_confidence"],
    ).copy()

if len(fp_rules_df) > 0:
    fp_rules_df = fp_rules_df[
        fp_rules_df["consequents"].apply(lambda s: len(s) == 1)
    ].copy()
    fp_rules_df["consequent_skill"] = fp_rules_df["consequents"].apply(lambda s: list(s)[0])
    fp_rules_df["rule_score"] = fp_rules_df["confidence"] * fp_rules_df["lift"]
    fp_rules_df = fp_rules_df.sort_values(
        ["rule_score", "confidence", "lift", "support"],
        ascending=False,
    ).reset_index(drop=True)

print("Association rules:", len(fp_rules_df))
display(fp_rules_df.head(10))

Frequent itemsets: 429643
Association rules: 2385401


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,consequent_skill,rule_score
0,"frozenset({Platform:Supabase, Language:JavaScript})",frozenset({Database:Supabase}),0.036527,0.048960,0.030026,0.822034,16.789761,1.0,0.028238,5.343937,0.976094,0.541395,0.812872,0.717656,Database:Supabase,13.801753
1,frozenset({Platform:Supabase}),frozenset({Database:Supabase}),0.041841,0.048960,0.033792,0.807645,16.495870,1.0,0.031744,4.944186,0.980399,0.592760,0.797742,0.748923,Database:Supabase,13.322805
2,"frozenset({Webframe:Ruby on Rails, Database:PostgreSQL, Language:JavaScript})",frozenset({Language:Ruby}),0.034566,0.068668,0.032038,0.926866,13.497699,1.0,0.029665,12.734534,0.959065,0.450000,0.921473,0.696716,Language:Ruby,12.510554
3,"frozenset({Webframe:Ruby on Rails, Language:HTML/CSS, Language:JavaScript})",frozenset({Language:Ruby}),0.034928,0.068668,0.032193,0.921713,13.422668,1.0,0.029795,11.896443,0.958994,0.450867,0.915941,0.695267,Language:Ruby,12.371854
4,"frozenset({Webframe:Ruby on Rails, Language:SQL})",frozenset({Language:Ruby}),0.034308,0.068668,0.031522,0.918797,13.380197,1.0,0.029167,11.469176,0.958135,0.441155,0.912810,0.688925,Language:Ruby,12.293685
5,"frozenset({Webframe:Ruby on Rails, Language:JavaScript})",frozenset({Language:Ruby}),0.041273,0.068668,0.037868,0.917500,13.361309,1.0,0.035034,11.288868,0.964985,0.525412,0.911417,0.734483,Language:Ruby,12.259001
6,"frozenset({Webframe:Ruby on Rails, Database:PostgreSQL})",frozenset({Language:Ruby}),0.040190,0.068668,0.036836,0.916560,13.347616,1.0,0.034077,11.161651,0.963816,0.511461,0.910408,0.726499,Language:Ruby,12.233887
7,"frozenset({Webframe:Ruby on Rails, Language:HTML/CSS})",frozenset({Language:Ruby}),0.037146,0.068668,0.033896,0.912500,13.288495,1.0,0.031345,10.643789,0.960423,0.471306,0.906048,0.703057,Language:Ruby,12.125752
8,"frozenset({Database:Supabase, Language:JavaScript})",frozenset({Platform:Supabase}),0.042408,0.041841,0.030026,0.708029,16.921985,1.0,0.028252,3.281695,0.982575,0.553758,0.695279,0.712831,Platform:Supabase,11.981260
9,"frozenset({Webframe:Ruby on Rails, Platform:Docker})",frozenset({Language:Ruby}),0.037198,0.068668,0.033689,0.905687,13.189273,1.0,0.031135,9.874854,0.959886,0.466762,0.898733,0.698148,Language:Ruby,11.945347


## 10. Build optimized FP rule index

In [10]:
def build_rule_index_by_len(rules_df):
    rule_index = defaultdict(dict)
    max_antecedent_len = 0

    if rules_df is None or len(rules_df) == 0:
        return rule_index, max_antecedent_len

    for _, row in rules_df.iterrows():
        antecedent = frozenset(row["antecedents"])
        antecedent_len = len(antecedent)
        max_antecedent_len = max(max_antecedent_len, antecedent_len)

        if antecedent not in rule_index[antecedent_len]:
            rule_index[antecedent_len][antecedent] = []

        rule_index[antecedent_len][antecedent].append({
            "skill": row["consequent_skill"],
            "rule_score": float(row["rule_score"]),
            "support": float(row["support"]),
            "confidence": float(row["confidence"]),
            "lift": float(row["lift"]),
        })

    return rule_index, max_antecedent_len

rule_index_by_len, rule_index_max_len = build_rule_index_by_len(fp_rules_df)
print("Rule buckets:", {k: len(v) for k, v in rule_index_by_len.items()})
print("Max antecedent length in rules:", rule_index_max_len)

Rule buckets: {2: 2593, 1: 130, 3: 16599, 5: 87838, 4: 50832, 6: 92549, 7: 61569, 8: 25613, 9: 6175, 10: 665}
Max antecedent length in rules: 10


## 11. FP-Growth recommendation

In [11]:
def recommend_fpgrowth(user_have, rule_index_by_len, k=10, max_ant_len=3):
    seen_skills = set(user_have)
    user_skill_list = sorted(seen_skills)
    candidate_scores = {}

    if len(user_skill_list) == 0:
        return []

    effective_max_len = min(max_ant_len, len(user_skill_list))

    for antecedent_len in range(1, effective_max_len + 1):
        bucket = rule_index_by_len.get(antecedent_len, {})
        if not bucket:
            continue

        for comb in combinations(user_skill_list, antecedent_len):
            antecedent = frozenset(comb)
            if antecedent not in bucket:
                continue

            for rule_info in bucket[antecedent]:
                skill = rule_info["skill"]
                if skill in seen_skills:
                    continue

                score = rule_info["rule_score"]
                old_score = candidate_scores.get(skill, -1.0)
                if score > old_score:
                    candidate_scores[skill] = score

    ranked = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    return [skill for skill, _ in ranked[:k]]

effective_fp_ant_len = min(CONFIG["fp_max_antecedent_len"], max(rule_index_max_len, 1))

for df in [valid_df, test_df]:
    df["pred_fp_10"] = df["have_list"].apply(
        lambda skills: recommend_fpgrowth(skills, rule_index_by_len, k=10, max_ant_len=effective_fp_ant_len)
    )
    df["pred_fp_5"] = df["pred_fp_10"].apply(lambda items: items[:5])

## 12. Build weighted skill graph

In [12]:
edge_a_col = "skill_a" if "skill_a" in edges_df.columns else ("source" if "source" in edges_df.columns else edges_df.columns[0])
edge_b_col = "skill_b" if "skill_b" in edges_df.columns else ("target" if "target" in edges_df.columns else edges_df.columns[1])
edge_w_col = "weight" if "weight" in edges_df.columns else edges_df.columns[2]

skill_graph = nx.Graph()

for _, row in edges_df.iterrows():
    skill_a = str(row[edge_a_col])
    skill_b = str(row[edge_b_col])
    weight = float(row[edge_w_col])

    if skill_a == skill_b:
        continue

    if skill_graph.has_edge(skill_a, skill_b):
        skill_graph[skill_a][skill_b]["weight"] += weight
    else:
        skill_graph.add_edge(skill_a, skill_b, weight=weight)

print("Graph nodes:", skill_graph.number_of_nodes())
print("Graph edges:", skill_graph.number_of_edges())

Graph nodes: 183
Graph edges: 16653


## 13. Deterministic biased random walks

In [13]:
def node2vec_transition_weight(graph, prev_node, current_node, next_node, p=1.0, q=1.0):
    base_weight = graph[current_node][next_node].get("weight", 1.0)

    if prev_node is None:
        return base_weight

    if next_node == prev_node:
        alpha = 1.0 / p
    elif graph.has_edge(next_node, prev_node):
        alpha = 1.0
    else:
        alpha = 1.0 / q

    return base_weight * alpha

def weighted_choice(items, weights, rng):
    total = sum(weights)
    if total <= 0:
        return rng.choice(items)

    threshold = rng.random() * total
    cumulative = 0.0
    for item, weight in zip(items, weights):
        cumulative += weight
        if cumulative >= threshold:
            return item
    return items[-1]

def generate_biased_walk(graph, start_node, walk_length=12, p=1.0, q=1.0, seed=42):
    rng = random.Random(seed)
    walk = [start_node]

    while len(walk) < walk_length:
        current_node = walk[-1]
        neighbors = sorted(list(graph.neighbors(current_node)))
        if len(neighbors) == 0:
            break

        if len(walk) == 1:
            weights = [graph[current_node][nbr].get("weight", 1.0) for nbr in neighbors]
            next_node = weighted_choice(neighbors, weights, rng)
        else:
            prev_node = walk[-2]
            weights = [
                node2vec_transition_weight(graph, prev_node, current_node, nbr, p=p, q=q)
                for nbr in neighbors
            ]
            next_node = weighted_choice(neighbors, weights, rng)

        walk.append(next_node)

    return walk

node_order = {node: idx for idx, node in enumerate(sorted(skill_graph.nodes()))}

def generate_walk_corpus(graph, num_walks=20, walk_length=12, p=1.0, q=1.0, seed=42):
    walk_corpus = []
    ordered_nodes = sorted(graph.nodes())
    shuffle_rng = random.Random(seed)

    for walk_round in range(num_walks):
        shuffled_nodes = ordered_nodes.copy()
        shuffle_rng.shuffle(shuffled_nodes)

        for start_node in shuffled_nodes:
            walk_seed = seed + walk_round * 100000 + node_order[start_node]
            walk = generate_biased_walk(
                graph,
                start_node,
                walk_length=walk_length,
                p=p,
                q=q,
                seed=walk_seed,
            )
            if len(walk) >= 2:
                walk_corpus.append(walk)

    return walk_corpus

walks = generate_walk_corpus(
    skill_graph,
    num_walks=CONFIG["n2v_num_walks"],
    walk_length=CONFIG["n2v_walk_length"],
    p=CONFIG["n2v_p"],
    q=CONFIG["n2v_q"],
    seed=RANDOM_SEED,
)

print("Number of walks:", len(walks))
print("Example walk:", walks[0][:10] if len(walks) > 0 else [])

Number of walks: 7320
Example walk: ['AIModel:Anthropic: Claude Sonnet', 'Language:C', 'Language:PowerShell', 'Platform:pnpm', 'Database:Redis', 'DevEnv:Claude Code', 'AIModel:DeepSeek (R- Reasoning models)', 'AIModel:openAI GPT (chatbot models)', 'AIModel:Anthropic: Claude Sonnet', 'Database:Microsoft SQL Server']


## 14. Train Word2Vec embeddings

In [14]:
skill_embedding_model = Word2Vec(
    sentences=walks,
    vector_size=CONFIG["n2v_dim"],
    window=CONFIG["n2v_window"],
    min_count=1,
    sg=1,
    workers=1,
    epochs=CONFIG["n2v_epochs"],
    seed=RANDOM_SEED,
)

print("Embedding vocabulary size:", len(skill_embedding_model.wv.index_to_key))

Embedding vocabulary size: 183


## 15. Similarity cache

In [15]:
def build_similarity_cache(model, topn=30):
    similarity_cache = {}
    vocab = model.wv.index_to_key
    topn = min(topn, max(len(vocab) - 1, 1))

    for skill in vocab:
        try:
            similarity_cache[skill] = model.wv.most_similar(skill, topn=topn)
        except KeyError:
            similarity_cache[skill] = []

    return similarity_cache

similarity_cache = build_similarity_cache(
    skill_embedding_model,
    topn=CONFIG["sim_cache_topn"],
)

print("Similarity cache size:", len(similarity_cache))

Similarity cache size: 183


## 16. Node2Vec recommendation

In [16]:
def recommend_node2vec_top10(user_have, similarity_cache):
    seen_skills = set(user_have)
    aggregate_scores = defaultdict(float)
    aggregate_counts = defaultdict(int)

    for skill in user_have:
        if skill not in similarity_cache:
            continue

        for other_skill, sim in similarity_cache[skill]:
            if other_skill in seen_skills:
                continue
            aggregate_scores[other_skill] += float(sim)
            aggregate_counts[other_skill] += 1

    for skill in list(aggregate_scores.keys()):
        aggregate_scores[skill] /= aggregate_counts[skill]

    ranked = sorted(aggregate_scores.items(), key=lambda x: x[1], reverse=True)
    return [skill for skill, _ in ranked[:10]]

for df in [valid_df, test_df]:
    df["pred_n2v_10"] = df["have_list"].apply(lambda skills: recommend_node2vec_top10(skills, similarity_cache))
    df["pred_n2v_5"] = df["pred_n2v_10"].apply(lambda items: items[:5])

## 17. Hybrid recommendation

In [17]:
def collect_rule_component(user_have, rule_index_by_len, max_ant_len=3):
    seen_skills = set(user_have)
    user_skill_list = sorted(seen_skills)
    rule_component = {}
    rule_support = {}

    if len(user_skill_list) == 0:
        return rule_component, rule_support

    effective_max_len = min(max_ant_len, len(user_skill_list))

    for antecedent_len in range(1, effective_max_len + 1):
        bucket = rule_index_by_len.get(antecedent_len, {})
        if not bucket:
            continue

        for comb in combinations(user_skill_list, antecedent_len):
            antecedent = frozenset(comb)
            if antecedent not in bucket:
                continue

            for rule_info in bucket[antecedent]:
                skill = rule_info["skill"]
                if skill in seen_skills:
                    continue

                score = rule_info["rule_score"]
                support = rule_info["support"]

                if skill not in rule_component or score > rule_component[skill]:
                    rule_component[skill] = score
                    rule_support[skill] = support

    return rule_component, rule_support

def collect_embedding_component(user_have, similarity_cache):
    seen_skills = set(user_have)
    embed_scores = defaultdict(float)
    embed_counts = defaultdict(int)

    for skill in user_have:
        if skill not in similarity_cache:
            continue

        for other_skill, sim in similarity_cache[skill]:
            if other_skill in seen_skills:
                continue
            embed_scores[other_skill] += float(sim)
            embed_counts[other_skill] += 1

    for skill in list(embed_scores.keys()):
        embed_scores[skill] /= embed_counts[skill]

    return dict(embed_scores)

def normalize_score_map(score_map):
    if not score_map:
        return {}
    values = list(score_map.values())
    min_value = min(values)
    max_value = max(values)
    if math.isclose(max_value, min_value):
        return {skill: 1.0 for skill in score_map}
    return {
        skill: (score - min_value) / (max_value - min_value)
        for skill, score in score_map.items()
    }

def recommend_hybrid(user_have, rule_index_by_len, similarity_cache, support_fraction_map, alpha=0.6, beta=0.4, k=10, max_ant_len=3):
    seen_skills = set(user_have)

    rule_component, rule_support = collect_rule_component(
        user_have,
        rule_index_by_len,
        max_ant_len=max_ant_len,
    )
    embed_component = collect_embedding_component(user_have, similarity_cache)

    # Rule scores and embedding similarities have different numeric scales.
    # Normalize inside each user-specific candidate pool before weighted fusion.
    normalized_rule_component = normalize_score_map(rule_component)
    normalized_embed_component = normalize_score_map(embed_component)

    candidate_pool = set(normalized_rule_component.keys()) | set(normalized_embed_component.keys())

    final_scores = {}
    for skill in candidate_pool:
        if skill in seen_skills:
            continue

        rule_score = normalized_rule_component.get(skill, 0.0)
        embed_score = normalized_embed_component.get(skill, 0.0)

        combined_score = alpha * rule_score + beta * embed_score

        # Light popularity penalty: discourages always recommending only dominant skills
        # while preserving relevance from the normalized fusion score.
        support_value = support_fraction_map.get(skill, rule_support.get(skill, 0.0))
        popularity_penalty = 1.0 + support_value
        final_scores[skill] = combined_score / max(popularity_penalty, 1e-9)

    ranked = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    return [skill for skill, _ in ranked[:k]]

for df in [valid_df, test_df]:
    df["pred_hybrid_10"] = df["have_list"].apply(
        lambda skills: recommend_hybrid(
            skills,
            rule_index_by_len,
            similarity_cache,
            skill_support_fraction,
            alpha=CONFIG["alpha"],
            beta=CONFIG["beta"],
            k=10,
            max_ant_len=effective_fp_ant_len,
        )
    )
    df["pred_hybrid_5"] = df["pred_hybrid_10"].apply(lambda items: items[:5])


## 18. Compare models

In [18]:
def compare_models(df, k=10):
    prediction_mapping = {
        "Popularity": f"pred_popularity_{k}",
        "FP-Growth only": f"pred_fp_{k}",
        "Node2Vec only": f"pred_n2v_{k}",
        "Hybrid": f"pred_hybrid_{k}",
    }

    result_rows = []
    for model_name, pred_col in prediction_mapping.items():
        metrics = evaluate_predictions(
            df,
            pred_col,
            k=k,
            catalog=skill_catalog,
            support_fraction_map=skill_support_fraction,
        )
        result_rows.append({"Model": model_name, **metrics})

    result_df = pd.DataFrame(result_rows)
    return result_df.sort_values(
        ["Recall@K", "HitRate@K", "CatalogCoverage", "Novelty@K"],
        ascending=False,
    ).reset_index(drop=True)

valid_compare_5 = compare_models(valid_df, k=5)
valid_compare_10 = compare_models(valid_df, k=10)
test_compare_5 = compare_models(test_df, k=5)
test_compare_10 = compare_models(test_df, k=10)

print("Validation @5")
display(valid_compare_5)

print("Validation @10")
display(valid_compare_10)

print("Test @5")
display(test_compare_5)

print("Test @10")
display(test_compare_10)


Validation @5


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K,Novelty@K,AvgSupport@K
0,Popularity,1630,0.099940,0.339877,0.196721,0.642577,1.072175,0.491216
1,FP-Growth only,1630,0.083064,0.282209,0.508197,0.639755,2.662745,0.179301
2,Hybrid,1630,0.073901,0.257669,0.825137,0.638773,2.861513,0.158341
3,Node2Vec only,1630,0.035016,0.139264,0.907104,0.708712,3.698915,0.140422


Validation @10


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K,Novelty@K,AvgSupport@K
0,Popularity,1630,0.160787,0.450920,0.306011,0.475706,1.362031,0.409451
1,FP-Growth only,1630,0.153810,0.436196,0.524590,0.442638,2.486023,0.208283
2,Hybrid,1630,0.133689,0.406748,0.961749,0.445031,2.854006,0.168417
3,Node2Vec only,1630,0.069099,0.247239,0.978142,0.486871,3.792719,0.140210


Test @5


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K,Novelty@K,AvgSupport@K
0,Popularity,1702,0.114852,0.379553,0.213115,0.645006,1.080260,0.488900
1,FP-Growth only,1702,0.083398,0.299647,0.513661,0.641011,2.660192,0.179567
2,Hybrid,1702,0.070763,0.272620,0.808743,0.638895,2.850522,0.158315
3,Node2Vec only,1702,0.027280,0.140423,0.939891,0.712573,3.686315,0.144025


Test @10


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K,Novelty@K,AvgSupport@K
0,Popularity,1702,0.180040,0.504113,0.289617,0.477027,1.371308,0.407127
1,FP-Growth only,1702,0.153131,0.467098,0.524590,0.442773,2.485639,0.208196
2,Hybrid,1702,0.127078,0.422444,0.956284,0.446886,2.837742,0.169713
3,Node2Vec only,1702,0.059452,0.265570,0.961749,0.485253,3.785220,0.142656


## 19. Junior-only comparison (optional)

In [19]:
junior_flag_col = None
for col in ["is_junior_0_3y", "is_junior"]:
    if col in valid_df.columns:
        junior_flag_col = col
        break

if junior_flag_col is not None:
    valid_junior_df = valid_df[valid_df[junior_flag_col] == True].copy()
    test_junior_df = test_df[test_df[junior_flag_col] == True].copy()

    print("Validation junior @10")
    display(compare_models(valid_junior_df, k=10))

    print("Test junior @10")
    display(compare_models(test_junior_df, k=10))
else:
    print("No junior flag column found.")

Validation junior @10


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K,Novelty@K,AvgSupport@K
0,Popularity,209,0.182653,0.540670,0.207650,0.464115,1.349036,0.412283
1,FP-Growth only,209,0.160904,0.478469,0.508197,0.435407,2.482755,0.207831
2,Hybrid,209,0.155464,0.492823,0.781421,0.441627,2.821457,0.171386
3,Node2Vec only,209,0.070915,0.301435,0.857923,0.481340,3.724871,0.147543


Test junior @10


,Model,UsersEvaluated,Recall@K,HitRate@K,CatalogCoverage,Diversity@K,Novelty@K,AvgSupport@K
0,Popularity,258,0.188006,0.589147,0.234973,0.468217,1.401545,0.399730
1,FP-Growth only,258,0.143267,0.511628,0.497268,0.444186,2.498248,0.205559
2,Hybrid,258,0.125411,0.457364,0.781421,0.447674,2.856979,0.169574
3,Node2Vec only,258,0.065706,0.368217,0.863388,0.482558,3.794675,0.145009


## 21. Alpha/Beta ablation for the hybrid model

This module shows how the hybrid behavior changes as the score shifts from rule-dominant to embedding-dominant. It provides direct evidence for the research claim that hybrid recommendation is a controlled balance rather than a black box.

In [20]:
def describe_hybrid_weight(alpha, beta):
    if alpha == 1.0 and beta == 0.0:
        return "normalized_rule_component_only"
    if alpha == 0.0 and beta == 1.0:
        return "normalized_embedding_component_only_not_node2vec_baseline"
    return f"hybrid_alpha_{alpha}_beta_{beta}"

def run_hybrid_ablation(eval_df, alpha_beta_grid, k_values=(5, 10)):
    result_frames = {}

    for k in k_values:
        result_rows = []

        for alpha, beta in alpha_beta_grid:
            pred_col = f"pred_hybrid_ablation_a{str(alpha).replace('.', '_')}_b{str(beta).replace('.', '_')}_10"

            eval_df[pred_col] = eval_df["have_list"].apply(
                lambda skills: recommend_hybrid(
                    skills,
                    rule_index_by_len,
                    similarity_cache,
                    skill_support_fraction,
                    alpha=alpha,
                    beta=beta,
                    k=10,
                    max_ant_len=effective_fp_ant_len,
                )
            )

            metrics = evaluate_predictions(
                eval_df,
                pred_col,
                k=k,
                catalog=skill_catalog,
                support_fraction_map=skill_support_fraction,
            )
            result_rows.append({
                "FusionSetting": describe_hybrid_weight(alpha, beta),
                "alpha": alpha,
                "beta": beta,
                "UsersEvaluated": metrics["UsersEvaluated"],
                f"Recall@{k}": metrics["Recall@K"],
                f"HitRate@{k}": metrics["HitRate@K"],
                f"CatalogCoverage@{k}": metrics["CatalogCoverage"],
                f"Diversity@{k}": metrics["Diversity@K"],
                f"Novelty@{k}": metrics["Novelty@K"],
                f"AvgSupport@{k}": metrics["AvgSupport@K"],
            })

        result_frames[k] = pd.DataFrame(result_rows).sort_values(
            [f"Recall@{k}", f"HitRate@{k}", f"CatalogCoverage@{k}"],
            ascending=False,
        ).reset_index(drop=True)

    return result_frames

ablation_valid_results = run_hybrid_ablation(
    valid_df.copy(),
    alpha_beta_grid=ANALYSIS_CONFIG["ablation_grid"],
    k_values=(5, 10),
)

ablation_test_results = run_hybrid_ablation(
    test_df.copy(),
    alpha_beta_grid=ANALYSIS_CONFIG["ablation_grid"],
    k_values=(5, 10),
)

print("Validation ablation @5")
display(ablation_valid_results[5])

print("Validation ablation @10")
display(ablation_valid_results[10])

print("Test ablation @5")
display(ablation_test_results[5])

print("Test ablation @10")
display(ablation_test_results[10])


Validation ablation @5


,FusionSetting,alpha,beta,UsersEvaluated,Recall@5,HitRate@5,CatalogCoverage@5,Diversity@5,Novelty@5,AvgSupport@5
0,hybrid_alpha_0.8_beta_0.2,0.8,0.2,1630,0.081706,0.277301,0.633880,0.631656,2.784922,0.161754
1,normalized_rule_component_only,1.0,0.0,1630,0.081192,0.280368,0.540984,0.633865,2.788384,0.161281
2,hybrid_alpha_0.6_beta_0.4,0.6,0.4,1630,0.073901,0.257669,0.825137,0.638773,2.861513,0.158341
3,hybrid_alpha_0.4_beta_0.6,0.4,0.6,1630,0.061503,0.220245,0.923497,0.663067,3.320661,0.134446
4,normalized_embedding_component_only_not_node2vec_baseline,0.0,1.0,1630,0.020927,0.092025,0.819672,0.706258,5.059981,0.053470


Validation ablation @10


,FusionSetting,alpha,beta,UsersEvaluated,Recall@10,HitRate@10,CatalogCoverage@10,Diversity@10,Novelty@10,AvgSupport@10
0,normalized_rule_component_only,1.0,0.0,1630,0.145718,0.419018,0.573770,0.439018,2.618213,0.185969
1,hybrid_alpha_0.8_beta_0.2,0.8,0.2,1630,0.142802,0.416564,0.759563,0.438221,2.636268,0.183446
2,hybrid_alpha_0.6_beta_0.4,0.6,0.4,1630,0.133689,0.406748,0.961749,0.445031,2.854006,0.168417
3,hybrid_alpha_0.4_beta_0.6,0.4,0.6,1630,0.107275,0.345399,0.967213,0.471411,3.506821,0.131001
4,normalized_embedding_component_only_not_node2vec_baseline,0.0,1.0,1630,0.041518,0.164417,0.912568,0.492822,5.120528,0.052967


Test ablation @5


,FusionSetting,alpha,beta,UsersEvaluated,Recall@5,HitRate@5,CatalogCoverage@5,Diversity@5,Novelty@5,AvgSupport@5
0,normalized_rule_component_only,1.0,0.0,1702,0.077970,0.285546,0.524590,0.631140,2.780069,0.162030
1,hybrid_alpha_0.8_beta_0.2,0.8,0.2,1702,0.076085,0.280846,0.562842,0.631962,2.772784,0.162867
2,hybrid_alpha_0.6_beta_0.4,0.6,0.4,1702,0.070763,0.272620,0.808743,0.638895,2.850522,0.158315
3,hybrid_alpha_0.4_beta_0.6,0.4,0.6,1702,0.053262,0.223267,0.939891,0.672855,3.285319,0.136186
4,normalized_embedding_component_only_not_node2vec_baseline,0.0,1.0,1702,0.017140,0.088719,0.852459,0.709636,5.062422,0.054077


Test ablation @10


,FusionSetting,alpha,beta,UsersEvaluated,Recall@10,HitRate@10,CatalogCoverage@10,Diversity@10,Novelty@10,AvgSupport@10
0,normalized_rule_component_only,1.0,0.0,1702,0.144728,0.454172,0.524590,0.435723,2.608095,0.187118
1,hybrid_alpha_0.8_beta_0.2,0.8,0.2,1702,0.137782,0.442421,0.732240,0.438837,2.620898,0.185099
2,hybrid_alpha_0.6_beta_0.4,0.6,0.4,1702,0.127078,0.422444,0.956284,0.446886,2.837742,0.169713
3,hybrid_alpha_0.4_beta_0.6,0.4,0.6,1702,0.101936,0.372503,0.972678,0.465746,3.490960,0.132323
4,normalized_embedding_component_only_not_node2vec_baseline,0.0,1.0,1702,0.037218,0.166275,0.923497,0.490306,5.124925,0.053109


## 22. Limited stability analysis across random seeds

This module reruns the embedding branch with a few different seeds and measures how much Node2Vec-only and Hybrid metrics vary. It is not a full formal stability study, but it gives evidence directly aligned with the Node2Vec weakness discussed in the proposal and related work.

In [21]:
def train_embedding_with_seed(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)

    seed_walks = generate_walk_corpus(
        skill_graph,
        num_walks=CONFIG["n2v_num_walks"],
        walk_length=CONFIG["n2v_walk_length"],
        p=CONFIG["n2v_p"],
        q=CONFIG["n2v_q"],
        seed=seed_value,
    )

    seed_model = Word2Vec(
        sentences=seed_walks,
        vector_size=CONFIG["n2v_dim"],
        window=CONFIG["n2v_window"],
        min_count=1,
        sg=1,
        workers=1,
        epochs=CONFIG["n2v_epochs"],
        seed=seed_value,
    )

    seed_similarity_cache = build_similarity_cache(
        seed_model,
        topn=CONFIG["sim_cache_topn"],
    )

    return seed_model, seed_similarity_cache

def run_seed_stability(eval_df, seeds, k_values=(5, 10)):
    stability_rows = []

    for seed_value in seeds:
        _, seed_similarity_cache = train_embedding_with_seed(seed_value)

        tmp_df = eval_df.copy()

        tmp_df["pred_n2v_seed_10"] = tmp_df["have_list"].apply(
            lambda skills: recommend_node2vec_top10(skills, seed_similarity_cache)
        )
        tmp_df["pred_hybrid_seed_10"] = tmp_df["have_list"].apply(
            lambda skills: recommend_hybrid(
                skills,
                rule_index_by_len,
                seed_similarity_cache,
                skill_support_fraction,
                alpha=CONFIG["alpha"],
                beta=CONFIG["beta"],
                k=10,
                max_ant_len=effective_fp_ant_len,
            )
        )

        for k in k_values:
            n2v_metrics = evaluate_predictions(tmp_df, "pred_n2v_seed_10", k=k, catalog=skill_catalog, support_fraction_map=skill_support_fraction)
            hybrid_metrics = evaluate_predictions(tmp_df, "pred_hybrid_seed_10", k=k, catalog=skill_catalog, support_fraction_map=skill_support_fraction)

            stability_rows.append({
                "seed": seed_value,
                "model": "Node2Vec only",
                "k": k,
                "UsersEvaluated": n2v_metrics["UsersEvaluated"],
                "Recall": n2v_metrics["Recall@K"],
                "HitRate": n2v_metrics["HitRate@K"],
                "CatalogCoverage": n2v_metrics["CatalogCoverage"],
                "Diversity": n2v_metrics["Diversity@K"],
                "Novelty": n2v_metrics["Novelty@K"],
                "AvgSupport": n2v_metrics["AvgSupport@K"],
            })
            stability_rows.append({
                "seed": seed_value,
                "model": "Hybrid",
                "k": k,
                "UsersEvaluated": hybrid_metrics["UsersEvaluated"],
                "Recall": hybrid_metrics["Recall@K"],
                "HitRate": hybrid_metrics["HitRate@K"],
                "CatalogCoverage": hybrid_metrics["CatalogCoverage"],
                "Diversity": hybrid_metrics["Diversity@K"],
                "Novelty": hybrid_metrics["Novelty@K"],
                "AvgSupport": hybrid_metrics["AvgSupport@K"],
            })

    stability_df = pd.DataFrame(stability_rows)

    summary_df = (
        stability_df
        .groupby(["model", "k"], as_index=False)
        .agg(
            Recall_mean=("Recall", "mean"),
            Recall_std=("Recall", "std"),
            HitRate_mean=("HitRate", "mean"),
            HitRate_std=("HitRate", "std"),
            CatalogCoverage_mean=("CatalogCoverage", "mean"),
            CatalogCoverage_std=("CatalogCoverage", "std"),
            Diversity_mean=("Diversity", "mean"),
            Diversity_std=("Diversity", "std"),
            Novelty_mean=("Novelty", "mean"),
            Novelty_std=("Novelty", "std"),
            AvgSupport_mean=("AvgSupport", "mean"),
            AvgSupport_std=("AvgSupport", "std"),
        )
        .sort_values(["k", "model"])
        .reset_index(drop=True)
    )

    return stability_df, summary_df

if ANALYSIS_CONFIG["run_stability_analysis"]:
    stability_runs_df, stability_summary_df = run_seed_stability(
        test_df.copy(),
        seeds=ANALYSIS_CONFIG["stability_seeds"],
        k_values=tuple(ANALYSIS_CONFIG["stability_k_values"]),
    )

    print("Per-seed stability results")
    display(stability_runs_df)

    print("Seed stability summary (mean ± std)")
    display(stability_summary_df)
else:
    stability_runs_df = pd.DataFrame()
    stability_summary_df = pd.DataFrame()
    print("Stability analysis skipped because ANALYSIS_CONFIG['run_stability_analysis'] is False.")

Per-seed stability results


,seed,model,k,UsersEvaluated,Recall,HitRate,CatalogCoverage,Diversity,Novelty,AvgSupport
0,42,Node2Vec only,5,1702,0.027280,0.140423,0.939891,0.712573,3.686315,0.144025
1,42,Hybrid,5,1702,0.070763,0.272620,0.808743,0.638895,2.850522,0.158315
2,42,Node2Vec only,10,1702,0.059452,0.265570,0.961749,0.485253,3.785220,0.142656
3,42,Hybrid,10,1702,0.127078,0.422444,0.956284,0.446886,2.837742,0.169713
4,123,Node2Vec only,5,1702,0.031752,0.151586,0.939891,0.709166,3.877045,0.120710
5,123,Hybrid,5,1702,0.073764,0.273796,0.852459,0.639953,2.888591,0.153677
6,123,Node2Vec only,10,1702,0.068246,0.283196,0.972678,0.490599,3.913749,0.127429
7,123,Hybrid,10,1702,0.127261,0.428320,0.945355,0.448707,2.885177,0.163576
8,999,Node2Vec only,5,1702,0.034551,0.168038,0.950820,0.698120,4.215896,0.119861
9,999,Hybrid,5,1702,0.070900,0.270270,0.836066,0.632550,2.886754,0.154591


Seed stability summary (mean ± std)


,model,k,Recall_mean,Recall_std,HitRate_mean,HitRate_std,CatalogCoverage_mean,CatalogCoverage_std,Diversity_mean,Diversity_std,Novelty_mean,Novelty_std,AvgSupport_mean,AvgSupport_std
0,Hybrid,5,0.071809,0.001694,0.272229,0.001795,0.832423,0.022084,0.637133,0.004004,2.875289,0.021469,0.155528,0.002457
1,Node2Vec only,5,0.031194,0.003668,0.153349,0.013891,0.943534,0.006310,0.706620,0.007556,3.926419,0.268220,0.128198,0.013713
2,Hybrid,10,0.130710,0.006133,0.429299,0.007393,0.941712,0.016694,0.446651,0.002183,2.875523,0.033998,0.165982,0.003276
3,Node2Vec only,10,0.066631,0.006523,0.282413,0.016465,0.967213,0.005464,0.486291,0.003895,3.973707,0.224552,0.132332,0.008945


## 23. Skill-level and qualitative recommendation analysis

This module creates evidence that is easier to discuss in the report: what each model recommends for selected skills, and how the outputs compare on representative user cases.

In [22]:
available_skill_set = set(skill_catalog)

skill_probe_rows = []
for probe_skill in ANALYSIS_CONFIG["skill_probe_list"]:
    if probe_skill not in available_skill_set:
        skill_probe_rows.append({
            "probe_skill": probe_skill,
            "status": "missing_from_catalog",
            "fp_top10": [],
            "n2v_top10": [],
            "hybrid_top10": [],
        })
        continue

    input_skills = [probe_skill]

    skill_probe_rows.append({
        "probe_skill": probe_skill,
        "status": "ok",
        "fp_top10": recommend_fpgrowth(
            input_skills,
            rule_index_by_len,
            k=10,
            max_ant_len=effective_fp_ant_len,
        ),
        "n2v_top10": recommend_node2vec_top10(
            input_skills,
            similarity_cache,
        ),
        "hybrid_top10": recommend_hybrid(
            input_skills,
            rule_index_by_len,
            similarity_cache,
            skill_support_fraction,
            alpha=CONFIG["alpha"],
            beta=CONFIG["beta"],
            k=10,
            max_ant_len=effective_fp_ant_len,
        ),
    })

skill_level_analysis_df = pd.DataFrame(skill_probe_rows)
display(skill_level_analysis_df)

,probe_skill,status,fp_top10,n2v_top10,hybrid_top10
0,Language:Python,ok,"[Platform:Pip, Platform:Docker, DevEnv:Visual Studio Code, Language:Bash/Shell (all shells), Language:SQL, Language:JavaScript, Database:PostgreSQL, Language:HTML/CSS, Platform:npm, AIModel:openAI...","[Platform:npm, Language:Bash/Shell (all shells), Language:JavaScript, Platform:Docker, Platform:IBM Cloud, Platform:Pip, Language:Delphi, Platform:Make, Database:Datomic, Webframe:Node.js]","[Platform:Pip, Language:Bash/Shell (all shells), Platform:Docker, Language:JavaScript, Platform:npm, Database:PostgreSQL, DevEnv:Visual Studio Code, Language:SQL, Platform:IBM Cloud, Language:Delphi]"
1,Language:JavaScript,ok,"[Language:HTML/CSS, Language:TypeScript, Language:SQL, Platform:npm, DevEnv:Visual Studio Code, Platform:Docker, Webframe:Node.js, Webframe:React, Database:PostgreSQL, Language:Python]","[Platform:npm, Language:Bash/Shell (all shells), Language:SQL, Platform:Microsoft Azure, Webframe:Node.js, Platform:Yarn, Database:PostgreSQL, DevEnv:Cline and/or Roo, Database:SQLite, Platform:Pip]","[Platform:npm, Language:HTML/CSS, Language:SQL, Webframe:Node.js, Language:Bash/Shell (all shells), Database:PostgreSQL, Language:TypeScript, Platform:Microsoft Azure, Language:Python, Platform:Do..."
2,Database:PostgreSQL,ok,"[Platform:Docker, Language:SQL, Language:JavaScript, Platform:npm, Language:HTML/CSS, DevEnv:Visual Studio Code, Language:TypeScript, Language:Python, Platform:Amazon Web Services (AWS), Database:...","[Webframe:Node.js, Language:Bash/Shell (all shells), Platform:Docker, Platform:Splunk, Database:Redis, Language:JavaScript, AIModel:Gemini (Flash general purpose models), Database:Clickhouse, Data...","[Platform:Docker, Webframe:Node.js, Language:JavaScript, Language:Bash/Shell (all shells), Database:Redis, Platform:npm, Language:SQL, DevEnv:Visual Studio Code, Platform:Splunk, Language:Python]"
3,Platform:Docker,ok,"[Platform:npm, Database:PostgreSQL, Language:JavaScript, DevEnv:Visual Studio Code, Language:SQL, Language:HTML/CSS, Language:Python, Language:Bash/Shell (all shells), Language:TypeScript, Platfor...","[Platform:Splunk, Language:Bash/Shell (all shells), Database:Redis, Database:PostgreSQL, DevEnv:Visual Studio Code, Language:TypeScript, Language:Go, Language:SQL, Platform:Webpack, Platform:Yarn]","[Database:PostgreSQL, Platform:npm, Language:Bash/Shell (all shells), DevEnv:Visual Studio Code, Language:TypeScript, Database:Redis, Language:SQL, Language:Python, Platform:Splunk, Language:JavaS..."


In [23]:
def overlap_count(pred_list, target_list, k=10):
    return len(set(pred_list[:k]) & set(target_list))

qualitative_cases_df = test_df[test_df["target_list"].apply(len) > 0].copy()

qualitative_cases_df["pop_hit_count_10"] = qualitative_cases_df["pred_popularity_10"].apply(lambda pred: 0)
qualitative_cases_df["fp_hit_count_10"] = qualitative_cases_df.apply(
    lambda row: overlap_count(row["pred_fp_10"], row["target_list"], k=10),
    axis=1,
)
qualitative_cases_df["n2v_hit_count_10"] = qualitative_cases_df.apply(
    lambda row: overlap_count(row["pred_n2v_10"], row["target_list"], k=10),
    axis=1,
)
qualitative_cases_df["hybrid_hit_count_10"] = qualitative_cases_df.apply(
    lambda row: overlap_count(row["pred_hybrid_10"], row["target_list"], k=10),
    axis=1,
)

if "pred_popularity_10" in qualitative_cases_df.columns:
    qualitative_cases_df["pop_hit_count_10"] = qualitative_cases_df.apply(
        lambda row: overlap_count(row["pred_popularity_10"], row["target_list"], k=10),
        axis=1,
    )

qualitative_cases_df["hybrid_minus_fp_hits"] = (
    qualitative_cases_df["hybrid_hit_count_10"] - qualitative_cases_df["fp_hit_count_10"]
)

qualitative_cases_df["hybrid_minus_n2v_hits"] = (
    qualitative_cases_df["hybrid_hit_count_10"] - qualitative_cases_df["n2v_hit_count_10"]
)

qualitative_cases_df = qualitative_cases_df.sort_values(
    ["hybrid_hit_count_10", "hybrid_minus_fp_hits", "hybrid_minus_n2v_hits"],
    ascending=False,
).head(ANALYSIS_CONFIG["qualitative_num_cases"]).copy()

qualitative_cases_df = qualitative_cases_df[[
    "ResponseId",
    "have_list",
    "target_list",
    "pred_popularity_10",
    "pred_fp_10",
    "pred_n2v_10",
    "pred_hybrid_10",
    "pop_hit_count_10",
    "fp_hit_count_10",
    "n2v_hit_count_10",
    "hybrid_hit_count_10",
    "hybrid_minus_fp_hits",
    "hybrid_minus_n2v_hits",
]]

display(qualitative_cases_df)

,ResponseId,have_list,target_list,pred_popularity_10,pred_fp_10,pred_n2v_10,pred_hybrid_10,pop_hit_count_10,fp_hit_count_10,n2v_hit_count_10,hybrid_hit_count_10,hybrid_minus_fp_hits,hybrid_minus_n2v_hits
780,41266,"[Language:Bash/Shell (all shells), Language:HTML/CSS, Language:Java, Language:JavaScript, Language:SQL, Language:TypeScript, Database:Microsoft SQL Server, Database:MySQL, Database:PostgreSQL, Pla...","[Language:C, Language:C++, Language:Go, Language:Kotlin, Language:Lua, Language:OCaml, Language:PowerShell, Language:Rust, Language:Zig, Database:Cloud Firestore, Database:Firebase Realtime Databa...","[Platform:Docker, Language:Python, Platform:Amazon Web Services (AWS), Database:SQLite, Platform:Pip, Language:C#, Database:Redis, DevEnv:Visual Studio, Platform:Kubernetes, AIModel:Anthropic: Cla...","[Webframe:Next.js, Platform:Maven (build tool), Webframe:ASP.NET, Webframe:Spring Boot, Webframe:Express, Language:Kotlin, Platform:NuGet, Language:PHP, DevEnv:Visual Studio, Webframe:ASP.NET Core]","[Platform:Pacman, Language:Ada, AIModel:DeepSeek (V- General purpose models), Database:Microsoft Access, Webframe:Blazor, Language:PowerShell, Platform:Docker, Platform:Ansible, Platform:Google Cl...","[Webframe:Next.js, Webframe:Express, Platform:Maven (build tool), Language:Kotlin, Webframe:ASP.NET, Webframe:Spring Boot, Database:MariaDB, Language:PHP, Language:PowerShell, Webframe:ASP.NET Core]",5,5,3,7,2,4
1829,4952,"[Language:Bash/Shell (all shells), Language:HTML/CSS, Language:JavaScript, Language:PHP, Language:Python, Language:SQL, Database:MariaDB, Database:MongoDB, Database:MySQL, Database:PostgreSQL, Pla...","[Language:C#, Language:C++, Language:COBOL, Language:Go, Language:Java, Language:Perl, Language:R, Language:Ruby, Language:Rust, Language:TypeScript, Language:VBA, Database:BigQuery, Database:Cass...","[Platform:Docker, Language:TypeScript, Platform:npm, AIModel:openAI GPT (chatbot models), Platform:Amazon Web Services (AWS), Database:SQLite, Language:C#, Language:Java, Database:Microsoft SQL Se...","[Webframe:Next.js, Platform:Composer, AIModel:Gemini (Pro Reasoning models), Webframe:Laravel, Platform:Vite, Webframe:FastAPI, Webframe:jQuery, Platform:Yarn, Platform:npm, DevEnv:PhpStorm]","[Platform:Microsoft Azure, Platform:npm, Platform:Docker, DevEnv:Bolt, Language:PowerShell, Language:R, DevEnv:WebStorm, Language:Java, Language:Lisp, Database:InfluxDB]","[Webframe:Next.js, Platform:Composer, AIModel:Gemini (Pro Reasoning models), Webframe:Laravel, DevEnv:PhpStorm, Platform:Vite, Platform:Yarn, Webframe:FastAPI, Platform:npm, Webframe:jQuery]",8,7,5,7,0,2
1345,559,"[Language:Ada, Language:Bash/Shell (all shells), Language:C, Language:C++, Language:HTML/CSS, Language:Java, Language:JavaScript, Language:PHP, Language:PowerShell, Language:Python, Language:SQL, ...","[Language:Assembly, Language:C#, Language:COBOL, Language:Dart, Language:Delphi, Language:Elixir, Language:Erlang, Language:F#, Language:Fortran, Language:GDScript, Language:Go, Language:Groovy, L...","[DevEnv:Visual Studio Code, Language:C#, DevEnv:Visual Studio, DevEnv:IntelliJ IDEA, DevEnv:Notepad++, Platform:Homebrew, DevEnv:Vim, Platform:Yarn, Database:MariaDB, Platform:Make]","[Platform:Composer, Platform:NuGet, Webframe:ASP.NET Core, Platform:Maven (build tool), Language:C#, Webframe:ASP.NET, DevEnv:IntelliJ IDEA, DevEnv:Visual Studio, Platform:Make, Database:MariaDB]","[Language:C#, AIModel:Alibaba Cloud Qwen models, DevEnv:Trae, Webframe:ASP.NET Core, Language:COBOL, Language:OCaml, Language:Mojo, Database:Datomic, Webframe:Deno, Platform:IBM Cloud]","[Platform:Composer, Platform:NuGet, Webframe:ASP.NET Core, Platform:Maven (build tool), Language:C#, Platform:Make, Webframe:ASP.NET, Database:MariaDB, DevEnv:IntelliJ IDEA, DevEnv:Visual Studio]",5,7,9,7,0,-2
1743,42881,"[Language:Bash/Shell (all shells), Language:Java, Language:Python, Language:Scala, Language:SQL, Database:Databricks SQL, Database:Dynamodb, Database:Elasticsearch, Database:

## 24. Export outputs for report writing

In [24]:
OUTPUT_DIR = os.path.join(DATA_DIR, "research_outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

sample_cases = test_df[test_df["target_list"].apply(len) > 0].copy().head(20)
sample_cases = sample_cases[[
    "ResponseId",
    "have_list",
    "target_list",
    "pred_popularity_10",
    "pred_fp_10",
    "pred_n2v_10",
    "pred_hybrid_10",
]]

valid_compare_5.to_csv(os.path.join(OUTPUT_DIR, "valid_model_comparison_at5.csv"), index=False)
valid_compare_10.to_csv(os.path.join(OUTPUT_DIR, "valid_model_comparison_at10.csv"), index=False)
test_compare_5.to_csv(os.path.join(OUTPUT_DIR, "test_model_comparison_at5.csv"), index=False)
test_compare_10.to_csv(os.path.join(OUTPUT_DIR, "test_model_comparison_at10.csv"), index=False)
sample_cases.to_csv(os.path.join(OUTPUT_DIR, "sample_test_recommendations.csv"), index=False)

if len(fp_rules_df) > 0:
    fp_rules_df.head(500).to_csv(os.path.join(OUTPUT_DIR, "best_fpgrowth_rules.csv"), index=False)
    fp_rules_df.to_csv(os.path.join(OUTPUT_DIR, "full_fpgrowth_rules.csv"), index=False)
else:
    pd.DataFrame().to_csv(os.path.join(OUTPUT_DIR, "best_fpgrowth_rules.csv"), index=False)
    pd.DataFrame().to_csv(os.path.join(OUTPUT_DIR, "full_fpgrowth_rules.csv"), index=False)

# Added research-strengthening outputs
ablation_valid_results[5].to_csv(os.path.join(OUTPUT_DIR, "ablation_valid_at5.csv"), index=False)
ablation_valid_results[10].to_csv(os.path.join(OUTPUT_DIR, "ablation_valid_at10.csv"), index=False)
ablation_test_results[5].to_csv(os.path.join(OUTPUT_DIR, "ablation_test_at5.csv"), index=False)
ablation_test_results[10].to_csv(os.path.join(OUTPUT_DIR, "ablation_test_at10.csv"), index=False)

if not stability_runs_df.empty:
    stability_runs_df.to_csv(os.path.join(OUTPUT_DIR, "stability_runs_test.csv"), index=False)
if not stability_summary_df.empty:
    stability_summary_df.to_csv(os.path.join(OUTPUT_DIR, "stability_summary_test.csv"), index=False)

skill_level_analysis_df.to_csv(os.path.join(OUTPUT_DIR, "skill_level_analysis.csv"), index=False)
qualitative_cases_df.to_csv(os.path.join(OUTPUT_DIR, "qualitative_case_analysis.csv"), index=False)

# Save reusable modeling artifacts for reproducibility/inference.
skill_embedding_model.save(os.path.join(OUTPUT_DIR, "skill_embedding_model.model"))
pd.DataFrame({"skill": sorted(skill_catalog)}).to_csv(os.path.join(OUTPUT_DIR, "skill_catalog_snapshot.csv"), index=False)
with open(os.path.join(OUTPUT_DIR, "experiment_config.json"), "w", encoding="utf-8") as f:
    json.dump({"CONFIG": CONFIG, "ANALYSIS_CONFIG": ANALYSIS_CONFIG}, f, indent=2, ensure_ascii=False)

# Lightweight report charts.
def save_metric_bar_chart(result_df, metric, title, filename):
    plot_df = result_df.sort_values(metric, ascending=False)
    plt.figure(figsize=(8, 4.5))
    plt.bar(plot_df["Model"], plot_df[metric])
    plt.title(title)
    plt.ylabel(metric)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=160)
    plt.close()

save_metric_bar_chart(test_compare_10, "Recall@K", "Test Recall@10 by Model", "test_recall_at10.png")
save_metric_bar_chart(test_compare_10, "HitRate@K", "Test HitRate@10 by Model", "test_hitrate_at10.png")
save_metric_bar_chart(test_compare_10, "CatalogCoverage", "Test Catalog Coverage@10 by Model", "test_catalog_coverage_at10.png")

print("Saved output files to:", OUTPUT_DIR)
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)


Saved output files to: d:\School\BTVN\BDM\fn\improved_notebook_copies\research_outputs
- ablation_test_at10.csv
- ablation_test_at5.csv
- ablation_valid_at10.csv
- ablation_valid_at5.csv
- best_fpgrowth_rules.csv
- experiment_config.json
- full_fpgrowth_rules.csv
- qualitative_case_analysis.csv
- sample_test_recommendations.csv
- skill_catalog_snapshot.csv
- skill_embedding_model.model
- skill_level_analysis.csv
- stability_runs_test.csv
- stability_summary_test.csv
- test_catalog_coverage_at10.png
- test_hitrate_at10.png
- test_model_comparison_at10.csv
- test_model_comparison_at5.csv
- test_recall_at10.png
- valid_model_comparison_at10.csv
- valid_model_comparison_at5.csv
